# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vedant-Jagtap/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN:
    print("✅ Hugging Face token loaded successfully!")
else:
    print("❌ HF_TOKEN not found.")

✅ Hugging Face token loaded successfully!


In [2]:
!pip -q install datasets huggingface_hub duckdb pandas pyarrow

In [3]:
from datasets import load_dataset

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_clients",
    split="train",
    token=HF_TOKEN
)

print(dataset)

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

dim_clients.parquet: reconstructing file:   0%|          |  0.00B / 3.38kB            

dim_clients.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/104 [00:00<?, ? examples/s]

Dataset({
    features: ['client_hash_id', 'is_active', 'has_gsc_access', 'has_ga4_access', 'access_profile', 'client_created_date', 'client_updated_date', 'gsc_data_start', 'ga4_data_start'],
    num_rows: 104
})


In [4]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN
)

for file in files:
    print(file)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [5]:
from huggingface_hub import hf_hub_download

readme_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="README.md",
    token=HF_TOKEN
)

with open(readme_path, "r", encoding="utf-8") as f:
    print(f.read())

---
license: other
language:
- en
tags:
- seo
- content-performance
- data-warehouse
- tabular
- education
- flyrank-internship
pretty_name: FlyRank Internship — Warehouse Star Schema (Pseudonymized, Gated)
size_categories:
- 10M<n<100M
extra_gated_prompt: >-
  By requesting access you agree to the FlyRank Internship Data Use Terms:
  anonymized research and education use only; no attempt to re-identify clients,
  domains, queries, keywords, or content; no redistribution of the raw data; and
  no client-identifying data in any public output (case study, repo, chart, or demo).
extra_gated_fields:
  Name: text
  Email: text
  Affiliation or cohort: text
  I agree to the FlyRank data-use terms: checkbox
configs:
- config_name: dim_clients
  data_files: dim_clients.parquet
- config_name: dim_content
  data_files: dim_content.parquet
- config_name: fact_content_daily_performance
  data_files: fact_content_daily_performance/**/*.parquet
- config_name: fact_content_query_90d
  data_files: fac

In [7]:
from datasets import load_dataset

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True,
    token=HF_TOKEN
)

first_row = next(iter(ds))

print(first_row.keys())

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

dict_keys(['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'])


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of Analysis

- **One row represents:** One content item (webpage) for one client on one report date.
- **Table used:** `fact_content_daily_performance`
- **Time window:** March 2026 (`month=2026-03`)
- **Prediction task:** Predict whether a webpage requires a content refresh.

In [8]:
from datasets import load_dataset

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True,
    token=HF_TOKEN
)

# Display one sample row
sample = next(iter(ds))

print("Sample row:")
for key, value in sample.items():
    print(f"{key}: {value}")

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Sample row:
report_date: 2025-01-27
client_hash_id: client_9958f0a7ae1df715
content_hash_id: content_3b70a18ea133b2bb
client_has_gsc: True
client_has_ga4: True
gsc_data_available: True
ga4_data_available: False
gsc_impressions: 30
gsc_clicks: 0
gsc_sum_position: 115
gsc_avg_position: 3.8333333333333335
ga4_pageviews: 0
ga4_sessions: 0
ga4_users: 0
ga4_engaged_sessions: 0
ga4_total_engagement_sec: 0
sessions_organic: 0
sessions_direct: 0
sessions_referral: 0
sessions_social: 0
sessions_paid: 0
sessions_ai: 0
ai_chatgpt: 0
ai_perplexity: 0
ai_gemini: 0
ai_copilot: 0
ai_claude: 0
ai_meta: 0
ai_other: 0
scroll_events: 0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Fields Classification

### Features
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_sessions

These are historical metrics that are available before making a prediction.

### Label / Proxy
- Content requires refresh (proxy created later from historical performance trends).

### Context
- report_date
- client_hash_id
- content_hash_id
- client_has_gsc
- client_has_ga4
- gsc_data_available
- ga4_data_available

These fields identify the record and describe data availability but are not direct predictive features.

### Excluded
- Future performance values
- Any label-derived columns created during experiments

Reason:
These would introduce target leakage because they would not be available at the time the prediction is made.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [13]:
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

march_data = """
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet
"""

print("DuckDB connected successfully!")

DuckDB connected successfully!


In [12]:
print(march_data)


hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet



In [15]:
import pandas as pd

# Load only March 2026 into a DataFrame
march_ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    data_files={"train": "fact_content_daily_performance/month=2026-03/data_0.parquet"},
    token=HF_TOKEN
)

march_df = march_ds.to_pandas()

print(march_df.shape)
march_df.head()

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

(9841378, 30)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
con.register("march_data", march_df)

print("Table registered successfully!")

Table registered successfully!


In [17]:
query1 = """
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS records
FROM march_data
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1;
"""

result1 = con.sql(query1).df()

print("Duplicate grain rows found:", len(result1))
result1.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain rows found: 0


,report_date,client_hash_id,content_hash_id,records


In [18]:
query2 = """
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM march_data;
"""

result2 = con.sql(query2).df()
result2

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [19]:
query3 = """
SELECT
    COUNT(*) AS rows_with_gsc
FROM march_data
WHERE gsc_data_available IS TRUE;
"""

result3 = con.sql(query3).df()
result3

,rows_with_gsc
0,3611061


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [20]:
print("""
Limitation:
This dataset cannot capture every factor affecting webpage performance.
Some clients have shorter historical data than others, and early records may only contain GSC data without GA4.
Therefore, the dataset may not fully represent long-term trends for every content page.
""")


Limitation:
This dataset cannot capture every factor affecting webpage performance.
Some clients have shorter historical data than others, and early records may only contain GSC data without GA4.
Therefore, the dataset may not fully represent long-term trends for every content page.



In [21]:
feature_df = march_df[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_pageviews",
        "sessions_organic"
    ]
]

print(feature_df.head())

   gsc_impressions  gsc_clicks  gsc_avg_position  ga4_pageviews  \
0               20           0          3.350000            NaN   
1                1           0          0.000000            NaN   
2              125           1          4.928000            NaN   
3                7           0          4.000000            NaN   
4               11           0          2.272727            NaN   

   sessions_organic  
0               NaN  
1               NaN  
2               NaN  
3               NaN  
4               NaN  


In [22]:
print("""
Feature 1: gsc_impressions
Available at prediction time because impressions are already recorded before deciding whether a page needs refresh.

Feature 2: gsc_clicks
Available because historical click data exists before making the prediction.

Feature 3: gsc_avg_position
Available because search ranking is known before the refresh decision.

Feature 4: ga4_pageviews
Available because page views are historical analytics collected before prediction.

Feature 5: sessions_organic
Available because past organic traffic is known before deciding whether to refresh content.
""")


Feature 1: gsc_impressions
Available at prediction time because impressions are already recorded before deciding whether a page needs refresh.

Feature 2: gsc_clicks
Available because historical click data exists before making the prediction.

Feature 3: gsc_avg_position
Available because search ranking is known before the refresh decision.

Feature 4: ga4_pageviews
Available because page views are historical analytics collected before prediction.

Feature 5: sessions_organic
Available because past organic traffic is known before deciding whether to refresh content.



In [25]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Use a smaller sample (50,000 rows)
sample_df = march_df.sample(n=50000, random_state=42).copy()

# Create a simple proxy label
sample_df["needs_refresh"] = (
    sample_df["gsc_clicks"] < sample_df["gsc_impressions"] * 0.02
).astype(int)

features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "sessions_organic",
]

X = sample_df[features].fillna(0)
y = sample_df["needs_refresh"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = RandomForestClassifier(
    n_estimators=50,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

honest_score = accuracy_score(y_test, pred)

print("Honest Accuracy:", round(honest_score, 4))

Honest Accuracy: 0.9994


In [26]:
# Create a copy with a leaked feature
X_leak = X.copy()

# Intentionally add the label as a feature
X_leak["leak_feature"] = y

X_train, X_test, y_train, y_test = train_test_split(
    X_leak,
    y,
    test_size=0.2,
    random_state=42
)

model = RandomForestClassifier(
    n_estimators=50,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

leak_score = accuracy_score(y_test, pred)

print("Accuracy with Leakage:", round(leak_score, 4))

# Remove the leaked feature
X_leak.drop(columns=["leak_feature"], inplace=True)

print("Leak feature removed.")

Accuracy with Leakage: 1.0
Leak feature removed.


## Self-check

Before you submit, confirm each line honestly:

- [✔] Every section above is filled — markdown thinking AND the code that backs it
- [✔] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔] No client names, URLs, or private queries anywhere
- [✔] My claims use careful words: observed, measured, directional, decision-support
- [✔] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.